# Pandas — Complete Revision Notebook (Data Wrangling)

A clean, topic-wise reference for Pandas: creating data, exploring it, selecting/filtering,
cleaning, transforming, grouping, and merging. All examples run on a small synthetic
dataset built below (a "students" table) so the notebook works standalone, with the
same operations you'd apply to any real CSV.

> Tip from your notes, kept here: most Pandas functions return a **new object** and leave
> the original unchanged, unless you pass `inplace=True`.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
print(pd.__version__)

3.0.2


## 1. Creating Series and DataFrames

- A **Series** is a single labeled 1-D column of data.
- A **DataFrame** is a 2-D table made of multiple Series sharing the same index.

In [2]:
# Series from a list
a = [1, 7, 2]
myvar = pd.Series(a)
print(myvar)

# Series with custom labels
myvar2 = pd.Series(a, index=["x", "y", "z"])
print(myvar2)

0    1
1    7
2    2
dtype: int64
x    1
y    7
z    2
dtype: int64


In [3]:
# Series from a dictionary (keys become the index)
calories = {"day1": 420, "day2": 380, "day3": 390}
s = pd.Series(calories)
print(s)

# Only some keys
s2 = pd.Series(calories, index=["day1", "day2"])
print(s2)

day1    420
day2    380
day3    390
dtype: int64
day1    420
day2    380
dtype: int64


In [4]:
# DataFrame from a dict of lists
data = {
    "name": ["Ali", "Sara", "John"],
    "age": [20, 22, 23],
    "marks": [85, 90, 78]
}
df = pd.DataFrame(data)
print(df)

   name  age  marks
0   Ali   20     85
1  Sara   22     90
2  John   23     78


In [5]:
# DataFrame from a list of lists
data = [["Ali", 20, 85], ["Sara", 22, 90], ["John", 23, 78]]
df2 = pd.DataFrame(data, columns=["name", "age", "marks"])
print(df2)

# Empty DataFrame, columns added later (common when building tables incrementally)
df3 = pd.DataFrame(columns=["name", "age", "marks"])
df3["name"] = ["ali", "asad", "khan"]
print(df3)

   name  age  marks
0   Ali   20     85
1  Sara   22     90
2  John   23     78
   name  age marks
0   ali  NaN   NaN
1  asad  NaN   NaN
2  khan  NaN   NaN


## 2. Reading Data From Files

| Function | Reads |
|---|---|
| `pd.read_csv()` | CSV files |
| `pd.read_json()` | JSON files |
| `pd.read_excel()` | Excel files |
| `pd.read_sql()` | SQL query results |

```python
data = pd.read_csv('path/to/file.csv')
df = pd.read_json('data.json')
df = pd.read_excel('data.xlsx')
```
We'll build our own dataset below so this notebook runs without external files.

In [6]:
np.random.seed(0)
n = 12
data = pd.DataFrame({
    'student_id': range(1, n + 1),
    'name': ['Ali', 'Samin', 'Hoorane', 'Saad', 'Amin', 'Alizeey',
              'Siffat', 'Anoushey', 'Azzan', 'Qasim', 'Bilal', 'Mina'],
    'gpa': [3.2, 3.4, 3.1, 3.7, 3.4, 3.9, 3.5, 3.9, 3.0, 2.9, 3.6, np.nan],
    'study_hours': [2, np.nan, 3, 1, 5, 5, 3, 4, 5, 1, 4, 2],
    'credit_hours': [12, 16, 13, 14, 18, 20, 10, 20, 12, 20, 15, 14],
    'gender': ['M', 'M', 'F', 'M', 'M', 'F', 'M', 'F', 'M', 'M', 'F', 'F'],
    'category': ['Science', 'Arts', 'Science', 'Commerce', 'Science', 'Arts',
                 'Commerce', 'Science', 'Arts', 'Commerce', 'Science', 'Arts']
})
data

,student_id,name,gpa,study_hours,credit_hours,gender,category
0,1,Ali,3.2,2.0,12,M,Science
1,2,Samin,3.4,NaN,16,M,Arts
2,3,Hoorane,3.1,3.0,13,F,Science
3,4,Saad,3.7,1.0,14,M,Commerce
4,5,Amin,3.4,5.0,18,M,Science
5,6,Alizeey,3.9,5.0,20,F,Arts
6,7,Siffat,3.5,3.0,10,M,Commerce
7,8,Anoushey,3.9,4.0,20,F,Science
8,9,Azzan,3.0,5.0,12,M,Arts
9,10,Qasim,2.9,1.0,20,M,Commerce


## 3. Exploring a DataFrame

| Function | What it shows |
|---|---|
| `.head(n)` | first n rows (default 5) |
| `.tail(n)` | last n rows |
| `.shape` | (rows, columns) |
| `.info()` | column dtypes, non-null counts, memory usage |
| `.describe()` | summary stats for numeric columns |
| `.columns` / `.index` | column names / row labels |
| `.dtypes` | dtype of every column |

In [7]:
print(data.head())      # first 5 rows
print(data.tail(3))     # last 3 rows
print(data.shape)       # (rows, columns)

   student_id     name  gpa  study_hours  credit_hours gender  category
0           1      Ali  3.2          2.0            12      M   Science
1           2    Samin  3.4          NaN            16      M      Arts
2           3  Hoorane  3.1          3.0            13      F   Science
3           4     Saad  3.7          1.0            14      M  Commerce
4           5     Amin  3.4          5.0            18      M   Science
    student_id   name  gpa  study_hours  credit_hours gender  category
9           10  Qasim  2.9          1.0            20      M  Commerce
10          11  Bilal  3.6          4.0            15      F   Science
11          12   Mina  NaN          2.0            14      F      Arts
(12, 7)


In [8]:
data.info()   # dtypes + non-null counts -> first thing to run on any new dataset

<class 'pandas.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   student_id    12 non-null     int64  
 1   name          12 non-null     str    
 2   gpa           11 non-null     float64
 3   study_hours   11 non-null     float64
 4   credit_hours  12 non-null     int64  
 5   gender        12 non-null     str    
 6   category      12 non-null     str    
dtypes: float64(2), int64(2), str(3)
memory usage: 804.0 bytes


In [9]:
data.describe()   # count, mean, std, min, quartiles, max for numeric columns

,student_id,gpa,study_hours,credit_hours
count,12.000000,11.000000,11.000000,12.000000
mean,6.500000,3.418182,3.181818,15.333333
std,3.605551,0.342982,1.537412,3.472838
min,1.000000,2.900000,1.000000,10.000000
25%,3.750000,3.150000,2.000000,12.750000
50%,6.500000,3.400000,3.000000,14.500000
75%,9.250000,3.650000,4.500000,18.500000
max,12.000000,3.900000,5.000000,20.000000


In [10]:
print(data.columns.tolist())
print(data.dtypes)

['student_id', 'name', 'gpa', 'study_hours', 'credit_hours', 'gender', 'category']
student_id        int64
name                str
gpa             float64
study_hours     float64
credit_hours      int64
gender              str
category            str
dtype: object


## 4. Selecting Columns and Rows

- `df['col']` -> single column, returns a **Series**
- `df[['col1','col2']]` -> multiple columns, returns a **DataFrame**
- `df.iloc[]` -> select by **integer position**
- `df.loc[]` -> select by **label**
- Boolean masks -> filter rows by condition

In [11]:
gpa_col = data['gpa']
print(type(gpa_col))          # Series

subset = data[['name', 'gpa', 'gender']]
print(type(subset))           # DataFrame
subset

<class 'pandas.Series'>
<class 'pandas.DataFrame'>


,name,gpa,gender
0,Ali,3.2,M
1,Samin,3.4,M
2,Hoorane,3.1,F
3,Saad,3.7,M
4,Amin,3.4,M
5,Alizeey,3.9,F
6,Siffat,3.5,M
7,Anoushey,3.9,F
8,Azzan,3.0,M
9,Qasim,2.9,M


In [12]:
print(data.iloc[0])            # first row, by position
print(data.iloc[1:3])          # rows 1-2 (end exclusive)
print(data.iloc[[1, 4, 5]])    # fancy row selection
print(data.iloc[:, [1, 2]])    # all rows, columns at position 1 and 2
print(data.iloc[2:6, 1:3])     # row+column slice combined

student_id            1
name                Ali
gpa                 3.2
study_hours         2.0
credit_hours         12
gender                M
category        Science
Name: 0, dtype: object
   student_id     name  gpa  study_hours  credit_hours gender category
1           2    Samin  3.4          NaN            16      M     Arts
2           3  Hoorane  3.1          3.0            13      F  Science
   student_id     name  gpa  study_hours  credit_hours gender category
1           2    Samin  3.4          NaN            16      M     Arts
4           5     Amin  3.4          5.0            18      M  Science
5           6  Alizeey  3.9          5.0            20      F     Arts
        name  gpa
0        Ali  3.2
1      Samin  3.4
2    Hoorane  3.1
3       Saad  3.7
4       Amin  3.4
5    Alizeey  3.9
6     Siffat  3.5
7   Anoushey  3.9
8      Azzan  3.0
9      Qasim  2.9
10     Bilal  3.6
11      Mina  NaN
      name  gpa
2  Hoorane  3.1
3     Saad  3.7
4     Amin  3.4
5  Alizeey  3.

In [13]:
# loc uses LABELS (works the same as iloc here since index is default integers)
print(data.loc[0, 'name'])
print(data.loc[0:3, ['name', 'gpa']])   # loc's end IS inclusive (unlike iloc)

Ali
      name  gpa
0      Ali  3.2
1    Samin  3.4
2  Hoorane  3.1
3     Saad  3.7


In [14]:
# Boolean mask filtering
mask = data['gender'] == 'F'
print(data[mask])

# Multiple conditions -> use & / | with each condition in parentheses
mask1 = data['category'] == 'Science'
mask2 = data['gpa'] > 3.3
print(data[mask1 & mask2])

    student_id      name  gpa  study_hours  credit_hours gender category
2            3   Hoorane  3.1          3.0            13      F  Science
5            6   Alizeey  3.9          5.0            20      F     Arts
7            8  Anoushey  3.9          4.0            20      F  Science
10          11     Bilal  3.6          4.0            15      F  Science
11          12      Mina  NaN          2.0            14      F     Arts
    student_id      name  gpa  study_hours  credit_hours gender category
4            5      Amin  3.4          5.0            18      M  Science
7            8  Anoushey  3.9          4.0            20      F  Science
10          11     Bilal  3.6          4.0            15      F  Science


In [15]:
# isin() -> check membership against a list of values
print(data['category'].isin(['Science', 'Arts']))
print(data[data['category'].isin(['Science', 'Arts'])])

0      True
1      True
2      True
3     False
4      True
5      True
6     False
7      True
8      True
9     False
10     True
11     True
Name: category, dtype: bool
    student_id      name  gpa  study_hours  credit_hours gender category
0            1       Ali  3.2          2.0            12      M  Science
1            2     Samin  3.4          NaN            16      M     Arts
2            3   Hoorane  3.1          3.0            13      F  Science
4            5      Amin  3.4          5.0            18      M  Science
5            6   Alizeey  3.9          5.0            20      F     Arts
7            8  Anoushey  3.9          4.0            20      F  Science
8            9     Azzan  3.0          5.0            12      M     Arts
10          11     Bilal  3.6          4.0            15      F  Science
11          12      Mina  NaN          2.0            14      F     Arts


In [16]:
# query() -> readable alternative to boolean masks -- ADDED
print(data.query("gender == 'F' and gpa > 3.3"))

    student_id      name  gpa  study_hours  credit_hours gender category
5            6   Alizeey  3.9          5.0            20      F     Arts
7            8  Anoushey  3.9          4.0            20      F  Science
10          11     Bilal  3.6          4.0            15      F  Science


## 5. Frequencies and Unique Values

`value_counts()` is one of the most-used EDA (exploratory data analysis) functions —
it instantly shows class balance, which is critical before training a classifier
(imbalanced classes need special handling).

In [17]:
print(data['category'].value_counts())
print()
print(data['category'].value_counts(normalize=True))  # as proportions -- ADDED, useful for class-balance checks
print()
print(data['category'].unique())
print(data['category'].nunique())

category
Science     5
Arts        4
Commerce    3
Name: count, dtype: int64

category
Science     0.416667
Arts        0.333333
Commerce    0.250000
Name: proportion, dtype: float64

<StringArray>
['Science', 'Arts', 'Commerce']
Length: 3, dtype: str
3


## 6. Sorting

| Function | Purpose |
|---|---|
| `sort_values()` | sort rows by column value(s) |
| `sort_index()` | sort rows by index label |

In [18]:
print(data.sort_values('gpa').head())
print(data.sort_values('gpa', ascending=False).head())

# sort by multiple columns, mixed order
print(data.sort_values(['category', 'gpa'], ascending=[True, False]).head())

   student_id     name  gpa  study_hours  credit_hours gender  category
9          10    Qasim  2.9          1.0            20      M  Commerce
8           9    Azzan  3.0          5.0            12      M      Arts
2           3  Hoorane  3.1          3.0            13      F   Science
0           1      Ali  3.2          2.0            12      M   Science
1           2    Samin  3.4          NaN            16      M      Arts
    student_id      name  gpa  study_hours  credit_hours gender  category
5            6   Alizeey  3.9          5.0            20      F      Arts
7            8  Anoushey  3.9          4.0            20      F   Science
3            4      Saad  3.7          1.0            14      M  Commerce
10          11     Bilal  3.6          4.0            15      F   Science
6            7    Siffat  3.5          3.0            10      M  Commerce
    student_id     name  gpa  study_hours  credit_hours gender  category
5            6  Alizeey  3.9          5.0          

## 7. Handling Duplicates

`drop_duplicates()` removes repeated rows — important before training, since duplicate
rows can bias a model or leak between train/test splits.

In [19]:
dup_demo = pd.concat([data, data.iloc[[0, 1]]], ignore_index=True)
print("before:", dup_demo.shape)
print(dup_demo.duplicated().sum(), "duplicate rows detected")

cleaned = dup_demo.drop_duplicates()
print("after :", cleaned.shape)

# drop duplicates based on specific column(s) only
print(data.drop_duplicates(subset=['category'], keep='first'))

before: (14, 7)
2 duplicate rows detected
after : (12, 7)
   student_id   name  gpa  study_hours  credit_hours gender  category
0           1    Ali  3.2          2.0            12      M   Science
1           2  Samin  3.4          NaN            16      M      Arts
3           4   Saad  3.7          1.0            14      M  Commerce


## 8. Missing Data

Real-world data is messy. Handling `NaN` correctly is one of the most important
preprocessing steps before feeding data into any ML model.

| Function | Purpose |
|---|---|
| `isnull()` / `notnull()` | detect missing / non-missing values |
| `dropna()` | remove rows/columns with missing values |
| `fillna()` | fill missing values |
| `interpolate()` | fill by interpolating between known values — *added* |

In [20]:
print(data.isnull().sum())     # count of missing values per column -- always check this first

student_id      0
name            0
gpa             1
study_hours     1
credit_hours    0
gender          0
category        0
dtype: int64


In [21]:
print(data.dropna().shape)                     # drop any row with a NaN
print(data.dropna(subset=['gpa']).shape)        # drop only if 'gpa' is NaN
print(data.dropna(axis=1, how='any').shape)     # drop columns containing any NaN

(10, 7)
(11, 7)
(12, 5)


In [22]:
filled = data.copy()
filled['gpa'] = filled['gpa'].fillna(filled['gpa'].mean())          # fill with column mean
filled['study_hours'] = filled['study_hours'].fillna(0)             # fill with a constant
print(filled.isnull().sum())

student_id      0
name            0
gpa             0
study_hours     0
credit_hours    0
gender          0
category        0
dtype: int64


In [23]:
# forward-fill / back-fill -> common for time-series data
ts = pd.Series([1, np.nan, np.nan, 4, np.nan, 6])
print("ffill:", ts.ffill().tolist())
print("bfill:", ts.bfill().tolist())
print("interpolate:", ts.interpolate().tolist())   # -- ADDED, linear interpolation between known points

ffill:

 [1.0, 1.0, 1.0, 4.0, 4.0, 6.0]
bfill: [1.0, 4.0, 4.0, 4.0, 6.0, 6.0]
interpolate: [1.0, 2.0, 3.0, 4.0, 5.0, 6.0]


## 9. Modifying Data: rename, drop, apply, map, astype

`apply()` / `map()` let you run custom (row-wise or element-wise) logic — this is exactly
how you'd engineer new features for a model.

In [24]:
renamed = data.rename(columns={'gpa': 'GPA', 'name': 'student_name'})
print(renamed.columns.tolist())

dropped = data.drop('student_id', axis=1)     # drop a column
print(dropped.columns.tolist())
dropped_row = data.drop(0, axis=0)            # drop row with index 0
print(dropped_row.shape)

['student_id', 'student_name', 'GPA', 'study_hours', 'credit_hours', 'gender', 'category']


['name', 'gpa', 'study_hours', 'credit_hours', 'gender', 'category']
(11, 7)


In [25]:
# apply() row-wise (axis=1) -> feature engineering example
def workload_score(row):
    return row['study_hours'] * row['credit_hours']

data['workload_score'] = data.apply(workload_score, axis=1)
print(data[['name', 'study_hours', 'credit_hours', 'workload_score']].head())

      name  study_hours  credit_hours  workload_score
0      Ali          2.0            12            24.0
1    Samin          NaN            16             NaN
2  Hoorane          3.0            13            39.0
3     Saad          1.0            14            14.0
4     Amin          5.0            18            90.0


In [26]:
# apply() column-wise (default axis=0) and map() for element-wise mapping on a Series
print(data['gpa'].apply(lambda x: round(x, 1) if pd.notnull(x) else x).head())

gender_map = {'M': 'Male', 'F': 'Female'}
print(data['gender'].map(gender_map).head())

0    3.2
1    3.4
2    3.1
3    3.7
4    3.4
Name: gpa, dtype: float64
0      Male
1      Male
2    Female
3      Male
4      Male
Name: gender, dtype: str


In [27]:
# astype() -> convert dtype, e.g. before ML models that require numeric input
encoded_gender = data['gender'].map({'M': 0, 'F': 1}).astype(int)
print(encoded_gender.head())

0    0
1    0
2    1
3    0
4    0
Name: gender, dtype: int64


## 10. GroupBy, Aggregation, and Pivot Tables

`groupby()` splits data into groups, applies a function to each, and combines results —
the classic "split-apply-combine" pattern, used constantly for summarizing training data
by category.

In [28]:
grouped = data.groupby('category')
print(len(grouped))              # number of groups (unique categories)
print(grouped['gpa'].mean())     # average gpa per category
print(grouped.first())           # first row of each group

3
category
Arts        3.433333
Commerce    3.366667
Science     3.440000
Name: gpa, dtype: float64
          student_id   name  gpa  study_hours  credit_hours gender  \
category                                                             
Arts               2  Samin  3.4          5.0            16      M   
Commerce           4   Saad  3.7          1.0            14      M   
Science            1    Ali  3.2          2.0            12      M   

          workload_score  
category                  
Arts               100.0  
Commerce            14.0  
Science             24.0  


In [29]:
# agg() -> apply multiple aggregation functions at once
summary = data.groupby('category').agg(
    avg_gpa=('gpa', 'mean'),
    max_gpa=('gpa', 'max'),
    count=('gpa', 'count')
)
print(summary)

           avg_gpa  max_gpa  count
category                          
Arts      3.433333      3.9      3
Commerce  3.366667      3.7      3
Science   3.440000      3.9      5

In [30]:
# transform() -> returns a result with the SAME shape as the original (broadcast back)
# common for creating features like "deviation from group average"
data['category_avg_gpa'] = data.groupby('category')['gpa'].transform('mean')
print(data[['name', 'category', 'gpa', 'category_avg_gpa']].head())

      name  category  gpa  category_avg_gpa
0      Ali   Science  3.2          3.440000
1    Samin      Arts  3.4          3.433333
2  Hoorane   Science  3.1          3.440000
3     Saad  Commerce  3.7          3.366667
4     Amin   Science  3.4          3.440000


In [31]:
pivot = pd.pivot_table(data, values='gpa', index='category', columns='gender', aggfunc='mean')
print(pivot)

# crosstab -> frequency table between two categorical columns -- ADDED (very common in EDA)
print(pd.crosstab(data['category'], data['gender']))

gender           F         M
category                    
Arts      3.900000  3.200000
Commerce       NaN  3.366667
Science   3.533333  3.300000


gender    F  M
category      
Arts      2  2
Commerce  0  3
Science   3  2


## 11. Combining Data: merge, join, concat

| Function | Use |
|---|---|
| `pd.concat()` | stack DataFrames vertically or horizontally |
| `pd.merge()` | SQL-style join on key column(s) |
| `df.join()` | join on index |

In [32]:
scores = pd.DataFrame({
    'student_id': [1, 2, 3, 4, 99],
    'exam_score': [88, 76, 91, 65, 50]
})

print("inner join:\n", pd.merge(data, scores, on='student_id', how='inner')[['name', 'exam_score']])
print("left join (keeps all students):\n",
      pd.merge(data, scores, on='student_id', how='left')[['name', 'exam_score']].head())

inner join:
       name  exam_score
0      Ali          88
1    Samin          76
2  Hoorane          91
3     Saad          65
left join (keeps all students):
       name  exam_score
0      Ali        88.0
1    Samin        76.0
2  Hoorane        91.0
3     Saad        65.0
4     Amin         NaN


In [33]:
part1 = data.iloc[:5]
part2 = data.iloc[5:]
combined = pd.concat([part1, part2], ignore_index=True)
print(combined.shape)

(12, 9)


## 12. Index Operations

`set_index()` / `reset_index()` control which column acts as the row label. Setting a
meaningful index (like an ID) makes lookups with `.loc[]` fast and intuitive.

In [34]:
indexed = data.set_index('name')
print(indexed.loc['Ali'])

back = indexed.reset_index()     # restores default integer index, 'name' becomes a column again
print(back.head())

student_id                1
gpa                     3.2
study_hours             2.0
credit_hours             12
gender                    M
category            Science
workload_score         24.0
category_avg_gpa       3.44
Name: Ali, dtype: object


      name  student_id  gpa  study_hours  credit_hours gender  category  \
0      Ali           1  3.2          2.0            12      M   Science   
1    Samin           2  3.4          NaN            16      M      Arts   
2  Hoorane           3  3.1          3.0            13      F   Science   
3     Saad           4  3.7          1.0            14      M  Commerce   
4     Amin           5  3.4          5.0            18      M   Science   

   workload_score  category_avg_gpa  
0            24.0          3.440000  
1             NaN          3.433333  
2            39.0          3.440000  
3            14.0          3.366667  
4            90.0          3.440000  


## 13. String Operations (`.str` accessor) — *added, missing from the original notes*

Text cleaning is a huge part of real-world data prep (categorical features, NLP inputs).

In [35]:
names = data['name']
print(names.str.upper().head())
print(names.str.lower().head())
print(names.str.contains('a', case=False).head())   # boolean mask for substring match
print(names.str.len().head())                        # length of each string
print(names.str.replace('a', '@', case=False).head())

0        ALI
1      SAMIN
2    HOORANE
3       SAAD
4       AMIN
Name: name, dtype: str
0        ali
1      samin
2    hoorane
3       saad
4       amin
Name: name, dtype: str
0    True
1    True
2    True
3    True
4    True
Name: name, dtype: bool
0    3
1    5
2    7
3    4
4    4
Name: name, dtype: int64
0        @li
1      S@min
2    Hoor@ne
3       S@@d
4       @min
Name: name, dtype: str


## 14. Datetime Handling — *added, missing from the original notes*

Almost every real dataset has a date/time column. `pd.to_datetime()` converts text
into proper datetime objects so you can extract features like day-of-week, month,
or compute time differences — common feature-engineering steps for ML.

In [36]:
dates = pd.to_datetime(['2024-01-05', '2024-02-14', '2024-03-21'])
df_dates = pd.DataFrame({'date': dates, 'value': [10, 20, 30]})

df_dates['year'] = df_dates['date'].dt.year
df_dates['month'] = df_dates['date'].dt.month
df_dates['day_of_week'] = df_dates['date'].dt.day_name()
print(df_dates)

        date  value  year  month day_of_week
0 2024-01-05     10  2024      1      Friday
1 2024-02-14     20  2024      2   Wednesday
2 2024-03-21     30  2024      3    Thursday


## 15. Correlation

`.corr()` shows linear relationships between numeric columns — a key feature-selection
tool: highly correlated features may be redundant, and correlation with the target
variable hints at predictive power.

In [37]:
print(data[['gpa', 'study_hours', 'credit_hours', 'workload_score']].corr())
print()
print(data['gpa'].corr(data['study_hours']))

                     gpa  study_hours  credit_hours  workload_score
gpa             1.000000     0.282464      0.323290        0.481883
study_hours     0.282464     1.000000      0.187038        0.908827
credit_hours    0.323290     0.187038      1.000000        0.547252
workload_score  0.481883     0.908827      0.547252        1.000000



0.2824637329959888


## 16. Binning and One-Hot Encoding — *added, crucial for ML preprocessing*

- `pd.cut()` / `pd.qcut()` turn continuous numbers into categorical bins (e.g. GPA -> letter grade).
- `pd.get_dummies()` performs **one-hot encoding**, converting categorical columns into
  numeric 0/1 columns — required before most ML models can use categorical features.

In [38]:
data['gpa_band'] = pd.cut(data['gpa'], bins=[0, 3.0, 3.5, 4.0], labels=['Low', 'Medium', 'High'])
print(data[['name', 'gpa', 'gpa_band']].head())

      name  gpa gpa_band
0      Ali  3.2   Medium
1    Samin  3.4   Medium
2  Hoorane  3.1   Medium
3     Saad  3.7     High
4     Amin  3.4   Medium


In [39]:
one_hot = pd.get_dummies(data[['gender', 'category']], drop_first=False)
print(one_hot.head())

   gender_F  gender_M  category_Arts  category_Commerce  category_Science
0     False      True          False              False              True
1     False      True           True              False             False
2      True     False          False              False              True
3     False      True          False               True             False
4     False      True          False              False              True


## 17. Sampling and Outlier Detection — *added, crucial for ML*

- `df.sample()` — randomly sample rows, e.g. to create a quick train/test split or a
  smaller subset for fast experimentation.
- IQR-based outlier detection — a standard preprocessing step before training.

In [40]:
sample = data.sample(frac=0.5, random_state=42)   # 50% random sample, reproducible
print(sample.shape)

train = data.sample(frac=0.8, random_state=1)
test = data.drop(train.index)
print("train:", train.shape, "| test:", test.shape)
# Note: for real projects prefer sklearn.model_selection.train_test_split,
# which also supports stratified splitting by class label.

(6, 10)
train: (10, 10) | test: (2, 10)


In [41]:
Q1 = data['gpa'].quantile(0.25)
Q3 = data['gpa'].quantile(0.75)
IQR = Q3 - Q1
lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

outliers = data[(data['gpa'] < lower) | (data['gpa'] > upper)]
print(f"IQR bounds: [{lower:.2f}, {upper:.2f}]")
print("outlier rows:\n", outliers[['name', 'gpa']])

IQR bounds: [2.40, 4.40]
outlier rows:
 Empty DataFrame
Columns: [name, gpa]
Index: []


## Summary — Original Notes vs What Was Added

**Covered from your original notebook:** Series/DataFrame creation, reading CSVs,
`head/tail/shape/info/describe`, column & row selection (`loc`/`iloc`), boolean
filtering, `isin`, `value_counts`, sorting, `drop_duplicates`, `groupby`, `merge`,
pivot tables, `corr`, `rename`, `set_index`/`reset_index`, `dropna`/`fillna`, `apply`.

**Added because they're essential for ML/AI and were missing:**
1. `query()` — readable filtering
2. `value_counts(normalize=True)` — checking class balance before classification
3. `interpolate()`, `ffill`/`bfill` — smarter missing-value handling for sequential data
4. `agg()` with multiple named aggregations, `transform()`, `crosstab()`
5. `.str` accessor — text cleaning for categorical/NLP features
6. `.dt` accessor / `pd.to_datetime()` — datetime feature engineering
7. `pd.cut()` / `pd.get_dummies()` — binning and one-hot encoding, required before
   most models can consume categorical data
8. `df.sample()` + a manual train/test split (with a note pointing to
   `sklearn.model_selection.train_test_split` for real projects)
9. IQR-based outlier detection